# Construcción de Variables: Distancia a Puertos Marítimos Principales

Este notebook documenta y ejecuta el proceso de construcción de la base de datos `mecanismos_distancia_puertos.dta`.

**Objetivo:** Generar una medida a nivel de comuna (1970) que capture la cercanía al puerto marítimo principal más cercano, como proxy de exposición geográfica al comercio internacional.

**Puertos de referencia:**
- Valparaíso
- San Antonio
- San Vicente (Talcahuano)
- Antofagasta
- Iquique
- Puerto Montt

**Método de distancia:** Distancia euclidiana ajustada por latitud media (misma metodología que `Construccion_Variables_Represion.ipynb`). Esta aproximación es adecuada para las distancias involucradas en el territorio chileno continental.

**Supuestos adoptados:**
1. Las coordenadas de cada comuna provienen de la base unificada `base_unificada_ma_elecciones_comunas1970.csv` (centroides de los polígonos comunales de 1970).
2. Las coordenadas de los puertos corresponden a la ubicación del terminal portuario principal de cada ciudad, obtenidas de fuentes geográficas estándar (Google Maps / OpenStreetMap).
3. Se utiliza `comuna` como llave primaria para compatibilidad con la base principal del estudio.

In [25]:
import pandas as pd
import numpy as np
from math import radians, cos, sin, asin, sqrt
import warnings
warnings.filterwarnings('ignore')

# Rutas Base 
PATH_MA = '/Users/Angelo/Dropbox/Tesis 2026 ME/Procesos Tesis/Data/outcomes/elecciones/base_unificada_ma_elecciones_comunas1970.csv'
PATH_OUT = '/Users/Angelo/Dropbox/Tesis 2026 ME/Procesos Tesis/Data/outcomes/elecciones/mecanismos_distancia_puertos.dta'

# 1. Cargar base principal (279 comunas de 1970 con coordenadas)
df_ma = pd.read_csv(PATH_MA)
print(f"Base cargada: {len(df_ma)} comunas")
print(f"Columnas disponibles: {df_ma.columns.tolist()}")
print(f"Comunas con coordenadas: {df_ma[['latitud','longitud']].dropna().shape[0]} de {len(df_ma)}")

Base cargada: 279 comunas
Columnas disponibles: ['comuna', 'log_ma_1970', 'log_ma_1980', 'delta_log_ma_1970_1980', 'delta_70_80_th_1', 'delta_70_80_th_4', 'delta_70_80_th_3_8', 'delta_70_80_th_6', 'delta_70_80_th_10', 'delta_70_80_th_12', 'pob_1970', 'pob_1982', 'vsi_pleb88', 'vno_pleb88', 'total_inscritos_88', 'vtotal_pleb88', 'nulos_pleb88', 'blancos_pleb88', 'share_si_88', 'share_no_88', 'v_apruebo_22', 'v_rechazo_22', 'v_votos_en_blanco_22', 'v_votos_nulos_22', 'total_votos_22', 'share_apruebo_22', 'share_rechazo_22', 'share_votos_en_blanco_22', 'share_votos_nulos_22', 'v_apruebo_20', 'v_rechazo_20', 'v_totales_20', 'v_votos_en_blanco_20', 'v_votos_nulos_20', 'total_votos_20', 'share_apruebo_20', 'share_rechazo_20', 'share_totales_20', 'share_votos_en_blanco_20', 'share_votos_nulos_20', 'urbano_1992', 'rural_1992', 'pob_total_1992', 'share_rural_1992', 'region_actual', 'provincia_1970', 'departamento_1970', 'delta_log_ma_1970_1980_local', 'delta_log_ma_1980_2024_local', 'delta_log_

## 1. Definición de Puertos de Referencia

Se definen los 6 puertos marítimos principales de Chile que servirán como puntos de referencia para calcular la exposición al comercio internacional. Las coordenadas corresponden a la ubicación del terminal portuario principal de cada ciudad.

**Fuentes de coordenadas:**
- Valparaíso: Terminal Portuario de Valparaíso (-33.0472, -71.6127)
- San Antonio: Puerto de San Antonio (-33.5933, -71.6214)
- San Vicente: Terminal Portuario San Vicente, Talcahuano (-36.7285, -73.1314)
- Antofagasta: Puerto de Antofagasta (-23.6509, -70.3975)
- Iquique: Terminal Portuario de Iquique (-20.2133, -70.1500)
- Puerto Montt: Terminal Portuario de Puerto Montt (-41.4693, -72.9424)

In [ ]:
# Definición de puertos marítimos principales con coordenadas geográficas
PUERTOS = {
    'Valparaiso':   {'latitud': -33.0472, 'longitud': -71.6127},
    'San Antonio':  {'latitud': -33.5933, 'longitud': -71.6214},
    'San Vicente':  {'latitud': -36.7285, 'longitud': -73.1314},
    'Antofagasta':  {'latitud': -23.6509, 'longitud': -70.3975},
    'Iquique':      {'latitud': -20.2133, 'longitud': -70.1500},
    'Puerto Montt': {'latitud': -41.4693, 'longitud': -72.9424},
    'Punta Arenas': {'latitud': -53.1625, 'longitud': -70.9000},
    'Arica':        {'latitud': -18.4746, 'longitud': -70.3207}, 
    'Coquimbo':     {'latitud': -29.9533, 'longitud': -71.3394}, 
    'Valdivia':     {'latitud': -39.8142, 'longitud': -73.2459}, 
}

df_puertos = pd.DataFrame(PUERTOS).T.reset_index()
df_puertos.columns = ['puerto', 'latitud', 'longitud']
print("Puertos de referencia definidos:")
print(df_puertos.to_string(index=False))
print(f"\nTotal puertos: {len(df_puertos)}")


Puertos de referencia definidos:
      puerto  latitud  longitud
  Valparaiso -33.0472  -71.6127
 San Antonio -33.5933  -71.6214
 San Vicente -36.7285  -73.1314
 Antofagasta -23.6509  -70.3975
     Iquique -20.2133  -70.1500
Puerto Montt -41.4693  -72.9424
Punta Arenas -53.1625  -70.9000
       Arica -18.4746  -70.3207
    Coquimbo -29.9533  -71.3394
    Valdivia -39.8142  -73.2459

Total puertos: 10


## 2. Integración Espacial

Cargamos la base unificada de Market Access que contiene las coordenadas (`latitud`, `longitud`) de cada comuna de 1970. Verificamos que todas las comunas tengan coordenadas válidas para el cálculo de distancias.

In [27]:
# Extraer las columnas necesarias de la base principal
df_final = df_ma[['comuna', 'latitud', 'longitud']].copy()

# Verificar coordenadas faltantes
sin_coords = df_final[df_final['latitud'].isna() | df_final['longitud'].isna()]
print(f"Comunas sin coordenadas: {len(sin_coords)}")
if len(sin_coords) > 0:
    print("ADVERTENCIA: Las siguientes comunas no tienen coordenadas y quedarán sin asignación de puerto:")
    print(sin_coords['comuna'].tolist())
else:
    print("✓ Todas las comunas tienen coordenadas válidas.")

print(f"\nTotal comunas para cálculo: {len(df_final)}")

Comunas sin coordenadas: 0
✓ Todas las comunas tienen coordenadas válidas.

Total comunas para cálculo: 279


## 3. Cálculo de Distancias a Puertos

Para cada comuna, calculamos la distancia geográfica en kilómetros hacia **cada uno de los 6 puertos de referencia**, utilizando la misma función de distancia euclidiana ajustada por latitud que se emplea en `Construccion_Variables_Represion.ipynb`.

Luego, identificamos el **puerto más cercano** y su distancia correspondiente.

In [28]:
def euclidiana_km(lon1, lat1, lon2, lat2):
    """Calcula distancia euclidiana en km ajustando por la latitud media.
    Misma función utilizada en Construccion_Variables_Represion.ipynb."""
    # Convertimos a radianes para calcular el coseno de la latitud media
    lat_mid = radians((lat1 + lat2) / 2.0)
    
    # 1 grado de latitud equivale a ~111.32 km
    # 1 grado de longitud equivale a ~111.32 km * cos(latitud)
    dx = (lon2 - lon1) * 111.32 * cos(lat_mid)
    dy = (lat2 - lat1) * 111.32
    
    # Distancia Euclidiana (Teorema de Pitágoras)
    return np.sqrt(dx**2 + dy**2)

# Mapeo manual para forzar distancia 0 en las comunas que SON el puerto principal
# Nota: Antofagasta corresponde a AGUAS BLANCAS en la base de 1970
COMUNAS_PUERTO = {
    'VALPARAISO': 'Valparaiso',
    'SAN ANTONIO': 'San Antonio',
    'SAN VICENTE': 'San Vicente',
    'AGUAS BLANCAS': 'Antofagasta',
    'IQUIQUE': 'Iquique',
    'PUERTO MONTT': 'Puerto Montt',
    'MAGALLANES': 'Punta Arenas',
    'ARICA': 'Arica',        
    'VALDIVIA': 'Valdivia',    
    # Nota: Coquimbo no está en el index histórico de df_final
}


# Calcular distancia a CADA puerto para CADA comuna
for nombre_puerto, coords in PUERTOS.items():
    col_name = f"dist_{nombre_puerto.lower().replace(' ', '_')}_km"
    distancias = []
    for idx, row in df_final.iterrows():
        if pd.isna(row['latitud']) or pd.isna(row['longitud']):
            distancias.append(np.nan)
        else:
            # Si la comuna es exactamente la comuna del puerto, forzamos distancia 0
            if row['comuna'] in COMUNAS_PUERTO and COMUNAS_PUERTO[row['comuna']] == nombre_puerto:
                d = 0.0
            else:
                d = euclidiana_km(row['longitud'], row['latitud'], 
                                coords['longitud'], coords['latitud'])
            distancias.append(d)
    df_final[col_name] = distancias

# Identificar el puerto más cercano y su distancia
cols_dist = [c for c in df_final.columns if c.startswith('dist_') and c.endswith('_km')]
nombres_puertos = list(PUERTOS.keys())

puerto_cercano = []
distancia_min = []

for idx, row in df_final.iterrows():
    if any(pd.isna(row[c]) for c in cols_dist):
        puerto_cercano.append(np.nan)
        distancia_min.append(np.nan)
    else:
        dists = [row[c] for c in cols_dist]
        idx_min = np.argmin(dists)
        puerto_cercano.append(nombres_puertos[idx_min])
        distancia_min.append(dists[idx_min])

df_final['puerto_mas_cercano'] = puerto_cercano
df_final['distancia_puerto_km'] = distancia_min

# Variables derivadas para regresiones (análogas a las de represión)
df_final['distancia_puerto_100km'] = df_final['distancia_puerto_km'] / 100.0
df_final['cerca_puerto_50km'] = (df_final['distancia_puerto_km'] <= 50).astype(int)
df_final['cerca_puerto_100km'] = (df_final['distancia_puerto_km'] <= 100).astype(int)

# Las comunas sin datos de coordenadas quedan con valores nulos
df_final.loc[df_final['distancia_puerto_km'].isna(), 'cerca_puerto_50km'] = np.nan
df_final.loc[df_final['distancia_puerto_km'].isna(), 'cerca_puerto_100km'] = np.nan

# Verificación: ¿alguna comuna sin asignación de puerto?
sin_puerto = df_final[df_final['puerto_mas_cercano'].isna()]
if len(sin_puerto) == 0:
    print("✓ VERIFICACIÓN EXITOSA: Todas las comunas tienen un puerto asignado.")
else:
    print(f"ADVERTENCIA: {len(sin_puerto)} comunas sin asignación de puerto:")
    print(sin_puerto['comuna'].tolist())

print(f"\nDistribución de comunas por puerto más cercano:")
print(df_final['puerto_mas_cercano'].value_counts().to_string())
print(f"\nPrimeras 10 filas:")
print(df_final[['comuna', 'latitud', 'longitud', 'puerto_mas_cercano', 'distancia_puerto_km']].head(10).to_string())


✓ VERIFICACIÓN EXITOSA: Todas las comunas tienen un puerto asignado.

Distribución de comunas por puerto más cercano:
puerto_mas_cercano
San Antonio     80
San Vicente     66
Valparaiso      36
Valdivia        30
Puerto Montt    25
Coquimbo        16
Punta Arenas     9
Iquique          7
Antofagasta      6
Arica            4

Primeras 10 filas:
          comuna    latitud   longitud puerto_mas_cercano  distancia_puerto_km
0  AGUAS BLANCAS -24.310687 -69.505347        Antofagasta             0.000000
1          AISEN -45.869382 -73.743836       Puerto Montt           494.049984
2      ALGARROBO -33.331369 -71.614484        San Antonio            29.165261
3          ALHUE -34.047310 -71.100951        San Antonio            69.793234
4          ANCUD -41.999822 -73.829668       Puerto Montt            94.448055
5      ANDACOLLO -30.201303 -71.181520           Coquimbo            31.519701
6          ANGOL -37.750749 -72.782720        San Vicente           117.917700
7         ARAUCO -37.

## 3.1 Calculo de Distancia en términos de tiempo

In [29]:
import unicodedata
import re
import numpy as np

# 1. Definir la función de normalización que faltaba
def normalizar_nombre(txt):
    if pd.isna(txt): return ''
    txt = str(txt).strip().upper()
    txt = unicodedata.normalize('NFD', txt)
    txt = ''.join(c for c in txt if unicodedata.category(c) != 'Mn')
    txt = re.sub(r'[^A-Z ]', '', txt).strip()
    return txt

# ==============================================================================
# NUEVO: Cálculo de Tiempo de Viaje a Puertos usando Matriz OD 1970
# ==============================================================================
BASE_DISTANCIA = '/Users/Angelo/Dropbox/Tesis 2026 ME/Procesos Tesis/Data/distancia/'
matriz_1970 = BASE_DISTANCIA + 'matriz_OD_1970.csv'

# Diccionario inverso para alias (ya conocido)
MODERNO_A_HISTORICO = {
    'ANTOFAGASTA': 'AGUAS BLANCAS', 'PORVENIR': 'BAHIA INUTIL', 'TORTEL': 'BAKER', 'LO PRADO': 'BARRANCAS', 
    'PUTRE': 'BELEN', 'TALTAL': 'CATALINA', 'TORRES DEL PAINE': 'CERRO CASTILLO', 'CAMARONES': 'CODPA', 
    'CHILE CHICO': 'GENERAL CARRERA', 'HIJUELAS': 'LA CALERA', 'FUTRONO': 'LAGO RANGO', 'POZO ALMONTE': 'LAGUNAS', 
    'LLAILLAY': 'LLAYLLAY', 'PUNTA ARENAS': 'MAGALLANES', 'CANELA': 'MINCHA', 'LAGUNA BLANCA': 'MORRO CHICO', 
    'CABO DE HORNOS': 'NAVARINO', 'COLCHANE': 'NEGREIROS', 'HUARA': 'PISAGUA', 'QUILLECO': 'QUILECO', 
    'LITUECHE': 'ROSARIO', 'RIO HURTADO': 'SAMO ALTO', 'MARIA ELENA': 'TOCO', 'MACUL': 'NUNOA', 
    'PENALOLEN': 'NUNOA', 'SAN JOAQUIN': 'SAN MIGUEL', 'PEDRO AGUIRRE CERDA': 'SAN MIGUEL', 'LO ESPEJO': 'SAN MIGUEL', 
    'CERRILLOS': 'MAIPU', 'ESTACION CENTRAL': 'SANTIAGO', 'QUINTA NORMAL': 'SANTIAGO', 'RECOLETA': 'CONCHALI', 
    'INDEPENDENCIA': 'CONCHALI', 'HUECHURABA': 'CONCHALI', 'PUDAHUEL': 'BARRANCAS', 'CERRO NAVIA': 'BARRANCAS', 
    'LA FLORIDA': 'FLORIDA', 'EL BOSQUE': 'SAN BERNARDO', 'LA PINTANA': 'SAN BERNARDO', 'SAN RAMON': 'LA GRANJA', 
    'VITACURA': 'LAS CONDES', 'LO BARNECHEA': 'LAS CONDES', 'LA CISTERNA': 'LA CISTERNA'
}

# Cargar OD
df_od = pd.read_csv(matriz_1970)
df_od['origin_norm'] = df_od['origin_COMUNA'].apply(normalizar_nombre)
df_od['dest_norm'] = df_od['destination_COMUNA'].apply(normalizar_nombre)
df_od = df_od[(df_od['origin_norm'] != '') & (df_od['dest_norm'] != '')]
od_dict = dict(tuple(df_od.groupby('origin_norm')))

def buscar_alias_od(comuna_buscada):
    """Busca comuna en matriz OD, incluyendo excepciones ortográficas y alias."""
    if comuna_buscada == 'AISEN' and comuna_buscada not in od_dict: return 'AYSEN'
    if comuna_buscada == 'COIHAIQUE' and comuna_buscada not in od_dict: return 'COYHAIQUE'
    if comuna_buscada in od_dict: return comuna_buscada
    cands = [k for k, v in MODERNO_A_HISTORICO.items() if v == comuna_buscada]
    for c in cands:
        if c in od_dict: return c
    return None

# Los destinos (Puertos) exactos como existen en la matriz OD
PUERTOS_OD = ['VALPARAISO', 'SAN ANTONIO', 'TALCAHUANO', 'ANTOFAGASTA', 
              'IQUIQUE', 'PUERTO MONTT', 'PUNTA ARENAS', 'ARICA', 'COQUIMBO', 'VALDIVIA']

# Los puertos como fueron nombrados en df_final históricamente (para forzar viaje 0)
COMUNAS_PUERTO_NORM = ['VALPARAISO', 'SAN ANTONIO', 'SAN VICENTE', 'AGUAS BLANCAS', 
                       'IQUIQUE', 'PUERTO MONTT', 'MAGALLANES', 'ARICA', 'VALDIVIA']


tiempos_viaje = []

for idx, row in df_final.iterrows():
    c_norm = normalizar_nombre(row['comuna'])
    
    # 1. Si la comuna es donde está el puerto, el tiempo es 0
    if c_norm in COMUNAS_PUERTO_NORM:
        tiempos_viaje.append(0.0)
        continue
        
    origen_od = buscar_alias_od(c_norm)
    if origen_od is None:
        tiempos_viaje.append(np.nan)
        continue
        
    rutas_origen = od_dict[origen_od]
    
    # 2. Buscar rutas solo hacia los 6 puertos
    rutas_puertos = rutas_origen[rutas_origen['dest_norm'].isin(PUERTOS_OD)]
    
    if len(rutas_puertos) > 0:
        # Extraer el mínimo tiempo entre todos los puertos
        min_time_sec = rutas_puertos['time_min'].min()
        tiempos_viaje.append(min_time_sec / 3600.0)  # Convertir a horas
    else:
        tiempos_viaje.append(np.nan)

# Asignar la nueva variable
df_final['tiempo_puerto_1970_hrs'] = tiempos_viaje


## 4. Estadísticas Descriptivas y Exportación

Reportamos estadísticas descriptivas de la distancia al puerto más cercano y exportamos la base final como `.dta` para mergear con la base principal del estudio utilizando `comuna` como llave primaria.

In [30]:
# ── Estadísticas Descriptivas ──────────────────────────────────────────────
print("=" * 65)
print("ESTADÍSTICAS DESCRIPTIVAS: Distancia al Puerto Más Cercano (km)")
print("=" * 65)

dist_col = df_final['distancia_puerto_km'].dropna()

print(f"  N:          {len(dist_col)}")
print(f"  Media:      {dist_col.mean():.2f} km")
print(f"  Mediana:    {dist_col.median():.2f} km")
print(f"  Desv. Est.: {dist_col.std():.2f} km")
print(f"  Mínimo:     {dist_col.min():.2f} km")
print(f"  Máximo:     {dist_col.max():.2f} km")
print(f"  P10:        {dist_col.quantile(0.10):.2f} km")
print(f"  P25:        {dist_col.quantile(0.25):.2f} km")
print(f"  P75:        {dist_col.quantile(0.75):.2f} km")
print(f"  P90:        {dist_col.quantile(0.90):.2f} km")

print(f"\n  Comunas a ≤50 km de un puerto:  {int(df_final['cerca_puerto_50km'].sum())}")
print(f"  Comunas a ≤100 km de un puerto: {int(df_final['cerca_puerto_100km'].sum())}")

# ── Estadísticas por puerto ────────────────────────────────────────────────
print(f"\n{'=' * 65}")
print("DISTANCIA MEDIA POR PUERTO MÁS CERCANO")
print(f"{'=' * 65}")
for puerto in PUERTOS.keys():
    subset = df_final[df_final['puerto_mas_cercano'] == puerto]
    if len(subset) > 0:
        print(f"  {puerto:15s}: N={len(subset):3d}, Media={subset['distancia_puerto_km'].mean():7.1f} km, "
              f"Min={subset['distancia_puerto_km'].min():6.1f}, Max={subset['distancia_puerto_km'].max():7.1f}")

# ── Exportación ────────────────────────────────────────────────────────────
# Columnas de salida (análogas a las de Construccion_Variables_Represion)
columnas_salida = [
    'comuna', 'latitud', 'longitud',
    'puerto_mas_cercano', 'distancia_puerto_km', 'distancia_puerto_100km',
    'cerca_puerto_50km', 'cerca_puerto_100km',
    'dist_valparaiso_km', 'dist_san_antonio_km', 'dist_san_vicente_km',
    'dist_antofagasta_km', 'dist_iquique_km', 'dist_puerto_montt_km',
    'dist_punta_arenas_km', 'dist_arica_km', 'dist_coquimbo_km', 'dist_valdivia_km',
    'tiempo_puerto_1970_hrs'
]



df_export = df_final[columnas_salida].copy()

# Guardar como DTA
df_export.to_stata(PATH_OUT, write_index=False)
print(f"Base exportada exitosamente a: {PATH_OUT}")
print(f"Total observaciones: {len(df_export)}")
df_export.head()


ESTADÍSTICAS DESCRIPTIVAS: Distancia al Puerto Más Cercano (km)
  N:          279
  Media:      114.59 km
  Mediana:    101.69 km
  Desv. Est.: 80.75 km
  Mínimo:     0.00 km
  Máximo:     608.00 km
  P10:        35.53 km
  P25:        69.44 km
  P75:        144.68 km
  P90:        190.35 km

  Comunas a ≤50 km de un puerto:  46
  Comunas a ≤100 km de un puerto: 134

DISTANCIA MEDIA POR PUERTO MÁS CERCANO
  Valparaiso     : N= 36, Media=   75.5 km, Min=   0.0, Max=  179.0
  San Antonio    : N= 80, Media=  100.7 km, Min=   0.0, Max=  208.4
  San Vicente    : N= 66, Media=  115.8 km, Min=   0.0, Max=  235.6
  Antofagasta    : N=  6, Media=  155.7 km, Min=   0.0, Max=  300.9
  Iquique        : N=  7, Media=  122.5 km, Min=   0.0, Max=  210.3
  Puerto Montt   : N= 25, Media=  157.8 km, Min=   0.0, Max=  580.7
  Punta Arenas   : N=  9, Media=  216.1 km, Min=   0.0, Max=  608.0
  Arica          : N=  4, Media=   75.0 km, Min=   0.0, Max=  113.7
  Coquimbo       : N= 16, Media=  147.4 km, Min

,comuna,latitud,longitud,puerto_mas_cercano,distancia_puerto_km,distancia_puerto_100km,cerca_puerto_50km,cerca_puerto_100km,dist_valparaiso_km,dist_san_antonio_km,dist_san_vicente_km,dist_antofagasta_km,dist_iquique_km,dist_puerto_montt_km,dist_punta_arenas_km,dist_arica_km,dist_coquimbo_km,dist_valdivia_km,tiempo_puerto_1970_hrs
0,AGUAS BLANCAS,-24.310687,-69.505347,Antofagasta,0.000000,0.000000,1.0,1.0,994.087199,1053.697395,1425.415511,0.000000,460.930806,1936.929121,3214.066173,655.146873,653.887892,1761.558982,0.000000
1,AISEN,-45.869382,-73.743836,Puerto Montt,494.049984,4.940500,0.0,0.0,1439.070026,1378.600429,1018.851285,2492.223080,2875.657647,494.049984,837.482105,3066.597808,1784.318711,675.287113,40.774146
2,ALGARROBO,-33.331369,-71.614484,San Antonio,29.165261,0.291653,1.0,1.0,31.634100,29.165261,402.655184,1084.187718,1467.539100,913.494133,2208.361607,1658.922066,376.949203,736.259584,1.106415
3,ALHUE,-34.047310,-71.100951,San Antonio,69.793234,0.697932,0.0,1.0,121.033446,69.793234,350.770686,1159.359039,1542.881063,841.960806,2127.964642,1735.303113,456.300751,669.743601,3.336980
4,ANCUD,-41.999822,-73.829668,Puerto Montt,94.448055,0.944481,0.0,1.0,1015.644804,955.761637,589.872839,2067.680854,2450.503335,94.448055,1261.952152,2640.482360,1359.654257,248.211072,26.718162
